# Statistic of simulated data

TODO: write git version of last functional run!

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib notebook

In [ ]:
import os
import sys
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# PlatoSim libraries
import platosim.utilities as ut
from platosim.lightcurve   import LightCurve
from platosim.slurm        import workerOverview
from platosim.matplotlibrc import setup_notebook
setup_notebook()

In [ ]:
path = "/lhome/nicholas/software/workdir/kul20"

---
## VSC overview
---

In [ ]:
paramFile = f'{path}/cluster_fcam.data'

In [ ]:
workerLog = f'{path}/slurmFCAM_sim3/worker.pbs.log55215928'
paramFileNew = '/lhome/nicholas/cluster.retry'

workerOverview(workerLog, paramFile, ofile=paramFileNew, plot=True)

---
## Simulation statistics
---

In [ ]:
# lcs1 = LightCurve(f'{path}/P1/lightcurve', mode='multi')
# lcs5 = LightCurve(f'{path}/P5/lightcurve_without_maskupdate', mode='multi')

### Table of all simulated LCs

In [ ]:
# df1 = lcs1.stat_lcs_per_star()
# df1.head()

In [ ]:
# df5 = lcs5.stat_sims()
# df5.head()

Compared to the 19200 LCs per quarter (i.e. 38400 LCs for both quarters) this is:

P1 sample 76.5% complete

P5 sample 77.1% complete

### Missing simulations

In [ ]:
ds = pd.read_feather(f'{path}/input/starcat_P1_SPF_targets.ftr')
df = pd.read_feather(f'{path}/simcatP1.ftr')

In [ ]:

# Single star
df_star = df[df.ID == star]
# Number of ncams
ncams = ds.ncams.iloc[star-1]
df_star

In [ ]:
# START OF FUNCTION TO FIND MISSING SIMULATIONS

from platosim.simulation import Simulation
from platosim.utilities import errorcode

alpha, delta, kappa = ut.getPointingField('KUL20')

dn0 = pd.DataFrame()
dn1 = pd.DataFrame()
df0 = pd.DataFrame()
df1 = pd.DataFrame()
i = 0

# Loop over each star

for star in range(1,2):
    
    # Single star
    df_star = df[df.ID == star]
    # Number of ncams
    ncams = ds.ncams.iloc[star-1]
    
    # Loop over each quarter
    
    for Q in df_star.Q.unique():
        df_Q = df_star[df_star.Q == Q]

        # Check for missing simulations
        if len(df_Q) != ncams:
            
            # Add the number of missing simulations to df0
            i += 1
            N = ncams-len(df_Q)
            df1 = pd.DataFrame({'star': star, 'Q': Q, 'N': N}, index=[i])
            df0 = pd.concat([df0, df1])
            
            # Check observability for each group
            sim = Simulation('test')
            dex = np.zeros(4)
            for G in range(4):
                dex[G], _ = sim.getStarsWithinCameraGroup(np.array([ds.ra.iloc[star-1]]), 
                                                          np.array([ds.dec.iloc[star-1]]), 
                                                          alpha, delta, kappa, 
                                                          cameraGroup=G+1, quarter=Q)
            if N == 0 and :
                
                
#             if dex:
#                 dn1 = df_Q[df_Q.G == G]
#                 if len(dn1) != 6:
# #                         dn1 = pd.DataFrame()
# #                         dn1['ID']     = np.ones(6) * df_Q.ID.iloc[star-1]
# #                         dn1['group']  = np.ones(6) * G
# #                         dn1['camera'] = np.arange(1,7)
# #                         dn0 = pd.concat([dn0, dn1])
#                     print(f'QUARTER {Q}')
#                     print(dn1)
            
            
            
# Check how many simulations that are (potentially) missing
errorcode('warning', f'{df0.N.sum()} simulations are missing!')

In [ ]:
dn0

In [ ]:
(19200 - df0.N.sum())/ 19200 * 100

In [ ]:
df1_fail = ut.pdMergeRows(df0, df1, identical=False)
df1_fail = df1_fail[df1_fail.star == 1].sort_values(by=['group', 'camera', 'quarter'])
df1_fail

In [ ]:
df1_fail.where(df1_fail.group == 1)

### LCs per star and quarter

In [ ]:
lcs1.stat_lcsPerStar(quarters=[23,24], ofile='stat_simsPerStarP1.txt')

In [ ]:
lcs.stat_lcsPerStar(quarters=[23,24], ofile='stat_simsPerStarP5.txt')

## LCs per N-CAM

In [ ]:
from platosim.hpc import HPC 
hpc = HPC(cpus=8)
# hpc.run(script='platonium', project='kul20', paramFile=paramFile)

In [ ]:
statFile = '/lhome/nicholas/software/workdir/kul20/statistics.txt'
df = pd.read_csv(statFile)

In [ ]:
df = df.rename(columns={'1': 'star', '1.1':'group', '1.2':'camera', '24':'quarter', '1.3':'flag'})

In [ ]:
df[df.flag == 0]

Statistics for P5 Q23
Ncams	N_exp	N_sim	Diff
00    	0	11	11
06    	100	58	42
11    	0	1	1
12    	200	188	12
15    	0	1	1
18    	100	92	8
19    	0	1	1
24    	600	148	348	
1000 stars ->	7838/19200 = 40.8%

Statistics for P5 Q24
Ncams	N_exp	N_sim
00    	0	17
01    	0	1
06    	100	48
12    	200	186
17    	0	1
18    	100	95
24    	600	152
1000 stars ->	7896/19200 = 41.1%

Statistics for P1 Q23
Ncams	N_exp	N_sim	Diff
00		82		
01		1
02		3
04		4
05		14
06	100	151
07		7
08		3
09		1
10		11
11		45
12	200	254
13		5
14		4
15		3
16		7
17		15
18		87
19		3
20		5
21		5
22		12
23		49
24	600	217
1000 stars ->	13694/19200 = 71.3%

Statistics for P1 Q24
Ncams	N_exp	N_sim	Diff
00		7
01		2
02		2
03		1
04		1
05		12
06	100	139
07		14
08		8
09		3
10		11
11		44
12	200	220
13		7
14		3
15		5
16		15
17		25
18	100	90
19		15
20		7
21		7
22		27
23		70
24	600	253
1000 stars ->	15629/19200 = 81.4%